In [24]:
# initial prompt draft
# eval dataset
# feed through claude
# feed them through a grader (maybe out of 10)
# avg scores
# change prompt in some way and repeat above steps

from dotenv import load_dotenv
import os

load_dotenv()

# create an API client
# from anthropic import Anthropic
from openai import OpenAI

# client = Anthropic()
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OPENROUTER_API_KEY")
)

model = "anthropic/claude-opus-4.6"

def add_user_message(messages, text):
    user_msg = {"role": "user", "content": text}
    messages.append(user_msg)

def add_assitant_message(messages, text):
    assitant_msg = {"role": "assitant", "content": text}
    messages.append(assitant_msg)

def chat(messages):
    message = client.chat.completions.create(
        model=model,
        max_tokens=150,
        messages=messages,
        # stop=stop
    )
    return message.choices[0].message.content

In [25]:
# Function to grade a test case + output using a model
def grade_by_model(test_case, output):
    eval_prompt = f"""
You are an expert AWS code reviewer. Your task is to evaluate the following AI-generated solution.

Original Task:
<task>
{test_case["task"]}
</task>

Solution to Evaluate:
<solution>
{output}
</solution>

Output Format
Provide your evaluation as a structured JSON object with the following fields, in this specific order:
- "strengths": An array of 1-3 key strengths
- "weaknesses": An array of 1-3 key areas for improvement
- "reasoning": A concise explanation of your overall assessment
- "score": A number between 1-10

Respond with JSON. Keep your response concise and direct.
Example response shape:
{{
    "strengths": string[],
    "weaknesses": string[],
    "reasoning": string,
    "score": number
}}
    """

    messages = []
    add_user_message(messages, eval_prompt)
    add_assitant_message(messages, "```json")
    text = chat(messages)
    if "```json" in text:
        text = text.split("```json")[1].split("```")[0].strip()
    elif "```" in text:
        text = text.split("```")[1].split("```")[0].strip()
    # print(text)
    return json.loads(text)
    # return json.loads(eval_text)

In [28]:
from statistics import mean
def run_prompt(test_case):
    # merges the prompt and test case and returns the result
    prompt_v1 = f"""
    please solve the following task:
    {test_case["task"]}
    """
    messages=[]
    add_user_message(messages,prompt_v1)
    output = chat(messages)
    return output
    # pass


def run_test_case(test_case):
    # calls the run_prompt,then grades the result
    output = run_prompt(test_case)
    # todo grade
    model_eval = grade_by_model(test_case, output)
    score = model_eval["score"]
    reasoning = model_eval["reasoning"]
    # score = 10
    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }
    # pass

def run_eval(dataset):
    # loads the dataset and calls run_test_case for each case
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    average_score = mean(result["score"] for result in results)
    print(f"Average score across {len(results)} test cases: {average_score:.2f}")

    return results
    # pass

In [29]:
import json
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

print(json.dumps(results, indent=2))

JSONDecodeError: Unterminated string starting at: line 12 column 16 (char 590)